[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/calculus/c07_parabola.ipynb)

# Archimedes' impossible area: integration from scratch

Cut a parabola with a straight line. More than twenty-two centuries ago, Archimedes found the exact area of the
curved piece, with no algebra and no calculus. He trapped the answer from above and below until only one number was
left. That is still what an integral is.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button at the top) to run it. Run each cell in turn, from the top down. When a slider appears, drag it and watch.
Nothing can break: if something looks odd, run everything again from the top. It works in your browser (no install)
and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look: one style for every figure in this notebook. Run this cell first.
import math
from fractions import Fraction
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Polygon
from ipywidgets import interact, IntSlider, ToggleButtons

SLATE, SLATE2 = "#1d2925", "#25332e"        # the slate ground
CHALK, DIM, FAINT = "#ece9e0", "#7e8e88", "#33433e"
CORAL = "#ef8a68"                           # the one key thing in each figure
TEAL, LILAC = "#8fcfc4", "#c3a9e6"          # the curve · the strips and areas

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "legend.facecolor": SLATE2, "legend.edgecolor": DIM, "legend.labelcolor": CHALK, "legend.framealpha": 0.9,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

def f(x):
    """The curve of this notebook: y = x squared."""
    return x * x

def strip_total(n, tops, b=1.0):
    """Add up n strips under y = x² from 0 to b. Each strip is b/n wide; its height is the curve at the strip's
    left edge, right edge or middle (tops = "left", "right" or "middle")."""
    w = b / n
    shift = {"left": 0.0, "right": 1.0, "middle": 0.5}[tops]
    return sum(f((i + shift) * w) * w for i in range(n))

## 1 · Squeeze an area between two totals

Cut the area under y = x², from 0 to 1, into thin strips. If each strip is as tall as its **right** edge, the strips
poke out above the curve: the total is too big. As tall as its **left** edge, they fall short: too small. The true
area is caught between the two totals.

Drag **strips** to add more, and watch the two totals squeeze together. Then switch to **middle** tops.

In [ ]:
def squeeze(strips=4, tops="left and right"):
    n = strips
    xs = np.linspace(0, 1, 400)
    fig, ax = plt.subplots(figsize=(10, 5.2))
    w = 1 / n
    if tops == "left and right":
        for i in range(n):
            x0 = i * w
            ax.add_patch(Rectangle((x0, 0), w, f(x0 + w), facecolor=LILAC, alpha=0.18, edgecolor=LILAC,
                                   lw=0.8 if n < 60 else 0))
            ax.add_patch(Rectangle((x0, 0), w, f(x0), facecolor=LILAC, alpha=0.45, edgecolor="none"))
        big, small = strip_total(n, "right"), strip_total(n, "left")
        title = (f"{n} strips:  right tops {big:.4f}  ·  left tops {small:.4f}  ·  gap {big - small:.4f} = 1/{n}")
    else:
        for i in range(n):
            x0 = i * w
            ax.add_patch(Rectangle((x0, 0), w, f(x0 + w / 2), facecolor=LILAC, alpha=0.4, edgecolor=LILAC,
                                   lw=0.8 if n < 60 else 0))
        mid = strip_total(n, "middle")
        title = f"{n} strips, middle tops:  {mid:.4f}   (off by {abs(mid - 1 / 3):.4f})"
    ax.plot(xs, f(xs), color=TEAL, lw=3, label="y = x²")
    ax.axhline(1 / 3, color=CORAL, lw=1.6, ls="--", label="one third")
    ax.set_xlim(0, 1.02); ax.set_ylim(0, 1.05); ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_title(title, fontsize=14)
    ax.legend(loc="upper left")
    plt.show()

interact(squeeze, strips=IntSlider(value=4, min=2, max=100, step=2, description="strips"),
         tops=ToggleButtons(options=["left and right", "middle"], description="tops"));

**Notice.** The gap between the two totals is always exactly one strip wide: one over n. Make the strips thinner and
the gap shrinks to nothing, so only one number stays caught between them: one third. With middle tops, four strips
are already within about 0.005.

## 2 · The totals, strip count by strip count

The same two totals for many strip counts at once. The coral line is one third. The big total comes down, the small
one comes up.

In [ ]:
ns = np.array([2, 4, 10, 20, 50, 100, 200, 500, 1000])
right = np.array([strip_total(n, "right") for n in ns])
left = np.array([strip_total(n, "left") for n in ns])
fig, ax = plt.subplots(figsize=(10, 4.4))
ax.semilogx(ns, right, "o-", color=LILAC, label="right tops (too big)")
ax.semilogx(ns, left, "s-", color=TEAL, label="left tops (too small)")
ax.axhline(1 / 3, color=CORAL, lw=1.8, label="one third")
ax.set_xlabel("number of strips"); ax.set_ylabel("total"); ax.legend(loc="upper right")
ax.set_title("two totals closing in on one number"); plt.show()
print(f"{'strips':>7} {'right tops':>12} {'left tops':>12} {'gap':>10}")
for n, r, l in zip(ns, right, left):
    print(f"{n:7d} {r:12.7f} {l:12.7f} {r - l:10.7f}")

## 3 · Archimedes' triangles

Archimedes filled the curved piece between y = x² and the line y = 1 with triangles. The first has its base on the
line and its tip where the curve runs parallel to it: area 1. Each new layer of triangles fills the gaps the last
one left, and each layer is a quarter of the one before.

Drag **layers** and watch the running total creep towards four thirds.

In [ ]:
def triangles(layers=1):
    fig, ax = plt.subplots(figsize=(10, 5.2))
    xs = np.linspace(-1, 1, 400)
    ax.fill_between(xs, f(xs), 1, color=FAINT, alpha=0.6)
    total = Fraction(0)
    pieces = [(Fraction(-1), Fraction(1))]                 # each gap: its two ends on the curve
    for k in range(layers):
        nxt = []
        for a, b in pieces:
            m = (a + b) / 2                                # the tip: where the curve runs parallel to the chord
            pts = [(float(a), float(a * a)), (float(m), float(m * m)), (float(b), float(b * b))]
            ax.add_patch(Polygon(pts, closed=True, facecolor=LILAC, alpha=0.75 - 0.08 * k, edgecolor=SLATE, lw=0.8))
            total += abs(Fraction(1, 2) * (a * (m * m - b * b) + m * (b * b - a * a) + b * (a * a - m * m)))
            nxt += [(a, m), (m, b)]
        pieces = nxt
    ax.plot(xs, f(xs), color=TEAL, lw=3); ax.plot([-1, 1], [1, 1], color=CHALK, lw=2)
    ax.set_xlim(-1.1, 1.1); ax.set_ylim(-0.05, 1.12); ax.set_aspect("equal")
    ax.set_title(f"{layers} layer{'s' if layers > 1 else ''}:  total {float(total):.6f}   ·   four thirds = {4 / 3:.6f}")
    plt.show()

interact(triangles, layers=IntSlider(value=1, min=1, max=8, step=1, description="layers"));

**Notice.** The total grows by a quarter, then a sixteenth, then a sixty-fourth … and never passes 1.3333…

## 4 · His proof that stops

Greek geometers did not trust a process that never ends. So Archimedes stopped after any number of layers and added
one third of the last layer. The total is exactly four thirds, every time. Exact fractions, no rounding:

In [ ]:
print(f"{'layers':>7}  {'total':>22}  {'+ a third of the last':>22}")
for k in range(1, 9):
    terms = [Fraction(1, 4) ** j for j in range(k)]
    s = sum(terms)
    print(f"{k:7d}  {str(s):>22}  {str(s + terms[-1] / 3):>22}")
    assert s + terms[-1] / 3 == Fraction(4, 3)

**Notice.** The right-hand column never changes. So the curved piece is four thirds of its first triangle, and the
area under the curve from 0 to 1 (the unit square minus half of that piece) is 1 − ½ × 4/3 = **one third**: the same
number the strips squeezed.

## 5 · Check your answer: the limit of a sum

With n strips, each 1/n wide, the right-hand total is the sum of (i/n)² × 1/n for i from 1 to n. sympy (a free
algebra library) adds it up exactly and takes the limit as n grows. Change `b` to find the area from 0 to b, and
run the cell again.

In [ ]:
import sympy as sp
i, n = sp.symbols("i n", positive=True, integer=True)
b = 1                                                   # try 2 (the slip in the video), or 3 (the challenge)
total = sp.simplify(sp.summation((b * i / n) ** 2 * b / n, (i, 1, n)))
print("right-hand total with n strips:", sp.factor(total))
print("as n grows, it goes to:", sp.limit(total, n, sp.oo))
print("Archimedes' way: box minus half a segment:", sp.Rational(b * b ** 2) - sp.Rational(1, 2) * sp.Rational(4, 3) * b * b ** 2)

**Try it.**
1. The area under y = x² from 0 to 2. (Each strip is 2/n wide, not 1/n.)
2. The area under y = x² from 0 to 3. (The box around it is 3 by 9.)

<details><summary>Answers</summary>

1. 8/3. The box is 2 × 4 = 8; above the curve sits ½ × 4/3 × 8 = 16/3; 8 − 16/3 = 8/3.
2. 9. The box is 3 × 9 = 27; above the curve sits ½ × 4/3 × 27 = 18; 27 − 18 = 9.

</details>

## Sources
T. L. Heath, *The Works of Archimedes* (Cambridge, 1897): *Quadrature of the Parabola*, Props. 23–24, and *On
Spirals*, Prop. 10. T. L. Heath, *The Method of Archimedes* (Cambridge, 1912). MacTutor History of Mathematics:
*Archimedes*; *Riemann*. All retrieved 2026-10-24.